In [1]:
import os
import math
import shutil
import subprocess
import tempfile
from pathlib import Path
from typing import List, Sequence, Tuple, Optional


def _scale_to_int_matrix(submatrix: List[List[float]], scale: int) -> List[List[int]]:
    """Scale float travel times to integers for LKH (it expects integer weights)."""
    n = len(submatrix)
    W = [[0] * n for _ in range(n)]
    for i in range(n):
        for j in range(n):
            # LKH expects non-negative ints; rounding is standard.
            W[i][j] = int(round(submatrix[i][j] * scale))
    return W


def _write_tsplib_explicit_full_matrix(
    tsp_path: Path,
    name: str,
    W: List[List[int]],
) -> None:
    """
    Write a TSPLIB file with EXPLICIT FULL_MATRIX weights.
    Nodes are 1..n in the TSPLIB file (TSPLIB indexing).
    """
    n = len(W)
    lines = []
    lines.append(f"NAME: {name}")
    lines.append("TYPE: TSP")
    lines.append(f"DIMENSION: {n}")
    lines.append("EDGE_WEIGHT_TYPE: EXPLICIT")
    lines.append("EDGE_WEIGHT_FORMAT: FULL_MATRIX")
    lines.append("EDGE_WEIGHT_SECTION")

    for i in range(n):
        lines.append(" ".join(str(W[i][j]) for j in range(n)))

    lines.append("EOF")
    tsp_path.write_text("\n".join(lines), encoding="utf-8")


def _write_lkh_par(
    par_path: Path,
    problem_file: Path,
    tour_file: Path,
    runs: int = 10,
    trace_level: int = 0,
) -> None:
    """
    Minimal LKH parameter file.
    - PROBLEM_FILE points to the TSPLIB instance.
    - TOUR_FILE is where LKH writes the best tour it found.
    """
    lines = [
        f"PROBLEM_FILE = {problem_file.as_posix()}",
        f"TOUR_FILE = {tour_file.as_posix()}",
        f"RUNS = {runs}",
        f"TRACE_LEVEL = {trace_level}",
    ]
    par_path.write_text("\n".join(lines), encoding="utf-8")


def _find_lkh_executable(explicit_path: Optional[str] = None) -> str:
    """
    Find the LKH executable.
    You can pass LKH_PATH explicitly, otherwise we try:
      1) explicit_path
      2) env var LKH_PATH
      3) LKH in PATH
      4) LKH-3 in PATH
    """
    if explicit_path and shutil.which(explicit_path):
        return explicit_path

    env_path = os.environ.get("LKH_PATH")
    if env_path and shutil.which(env_path):
        return env_path

    for cand in ("LKH", "LKH-3", "lkh"):
        p = shutil.which(cand)
        if p:
            return p

    raise FileNotFoundError(
        "Could not find LKH executable. Install LKH/LKH-3 and either:\n"
        "- put it on PATH as LKH (or LKH-3), or\n"
        "- set environment variable LKH_PATH=/path/to/LKH, or\n"
        "- pass lkh_path=... to the solver."
    )


def _run_lkh(par_file: Path, lkh_path: Optional[str] = None, timeout_sec: int = 120) -> None:
    """Run LKH on the given parameter file."""
    exe = _find_lkh_executable(lkh_path)
    # LKH prints to stdout; we keep it quiet by default.
    subprocess.run([exe, par_file.as_posix()], check=True, timeout=timeout_sec)


def _read_tsplib_tour(tour_path: Path) -> List[int]:
    """
    Parse TOUR_SECTION from a TSPLIB tour file.
    Returns a list of 1-based node indices, ending when -1 encountered.
    """
    text = tour_path.read_text(encoding="utf-8", errors="ignore").splitlines()

    in_section = False
    tour_1based: List[int] = []

    for line in text:
        line = line.strip()
        if not line:
            continue
        if line.upper().startswith("TOUR_SECTION"):
            in_section = True
            continue
        if not in_section:
            continue

        # TSPLIB tour lines can have multiple ints
        parts = line.split()
        for p in parts:
            v = int(p)
            if v == -1:
                return tour_1based
            if v == 0:
                # Some tour files may use 0 to terminate; handle defensively.
                return tour_1based
            tour_1based.append(v)

    if not tour_1based:
        raise ValueError("Could not find TOUR_SECTION in LKH output tour file.")
    return tour_1based


def _rotate_cycle_to_start(cycle: List[int], start_node: int) -> List[int]:
    """
    Rotate a cycle list so it begins with start_node.
    cycle is a sequence of node ids (already in your original node ids).
    """
    if start_node not in cycle:
        raise ValueError("start_node not present in cycle; cannot rotate.")
    k = cycle.index(start_node)
    return cycle[k:] + cycle[:k]


def solve_trip_with_lkh(
    travel_time: List[List[float]],
    trip_nodes: Sequence[int],
    depot: int = 0,
    scale: int = 1000,
    runs: int = 10,
    lkh_path: Optional[str] = None,
    timeout_sec: int = 120,
) -> List[int]:
    """
    Solve a single trip ordering using LKH, given a travel time matrix.

    Inputs
    ------
    travel_time : NxN float matrix
        Full matrix over all nodes (including depot).
    trip_nodes : list/sequence of customer node ids (global ids)
        The customers that must be visited in this trip.
    depot : int
        Depot node id in travel_time.
    scale : int
        Multiply travel times by this before rounding to int for LKH.
    runs : int
        LKH RUNS parameter (more runs = potentially better, but slower).
    lkh_path : str | None
        Explicit path/name of LKH executable (optional).
    timeout_sec : int
        Per-trip timeout.

    Returns
    -------
    route : list[int]
        Route including depot start and end: [depot, ..., depot]
        Visiting all nodes in trip_nodes exactly once.
    """
    if depot in trip_nodes:
        raise ValueError("trip_nodes should NOT include the depot; pass customers only.")

    # Build local node list: depot + trip customers
    local_nodes = [depot] + list(trip_nodes)
    m = len(local_nodes)

    # Extract the submatrix for these nodes
    sub = [[0.0] * m for _ in range(m)]
    for i in range(m):
        for j in range(m):
            sub[i][j] = travel_time[local_nodes[i]][local_nodes[j]]

    W = _scale_to_int_matrix(sub, scale=scale)

    with tempfile.TemporaryDirectory() as tmpdir:
        tmp = Path(tmpdir)
        tsp_file = tmp / "trip.tsp"
        par_file = tmp / "trip.par"
        tour_file = tmp / "trip.tour"

        _write_tsplib_explicit_full_matrix(tsp_file, name="TRIP_TSP", W=W)
        _write_lkh_par(par_file, problem_file=tsp_file, tour_file=tour_file, runs=runs, trace_level=0)

        _run_lkh(par_file, lkh_path=lkh_path, timeout_sec=timeout_sec)

        # Tour is in TSPLIB node ids 1..m. Convert to local indices 0..m-1
        tour_1based = _read_tsplib_tour(tour_file)
        tour_local = [v - 1 for v in tour_1based]  # 0..m-1

        # Convert to your original/global node ids
        cycle_global = [local_nodes[idx] for idx in tour_local]

        # Rotate so the cycle starts at depot
        cycle_global = _rotate_cycle_to_start(cycle_global, start_node=depot)

        # Convert cycle to a depot-start, depot-end route.
        # Since it's a cycle, we can represent the trip as: depot -> ... -> depot
        route = cycle_global + [depot]

        # Sanity: ensure it visits exactly the right customers once
        visited_customers = [n for n in route if n != depot]
        if sorted(visited_customers) != sorted(trip_nodes):
            raise RuntimeError("LKH route does not match trip_nodes (unexpected parsing/format issue).")

        return route


def solve_all_trips_with_lkh(
    travel_time: List[List[float]],
    trips: Sequence[Sequence[int]],
    depot: int = 0,
    scale: int = 1000,
    runs: int = 10,
    lkh_path: Optional[str] = None,
    timeout_sec: int = 120,
) -> List[List[int]]:
    """
    Solve ordering for each trip independently with LKH.

    trips: list of trips, where each trip is a list of customer node ids.
    returns: list of routes, each [depot, ..., depot]
    """
    routes = []
    for trip in trips:
        route = solve_trip_with_lkh(
            travel_time=travel_time,
            trip_nodes=trip,
            depot=depot,
            scale=scale,
            runs=runs,
            lkh_path=lkh_path,
            timeout_sec=timeout_sec,
        )
        routes.append(route)
    return routes


# -------------------------
# Example usage
# -------------------------
if __name__ == "__main__":
    # Example: 1 depot (0) + 6 customers (1..6)
    # Replace this with your real NxN travel_time matrix.
    travel_time = [
        [0, 4.2, 3.1, 7.0, 6.4, 5.5, 8.0],
        [4.2, 0, 2.2, 3.3, 4.1, 6.0, 7.2],
        [3.1, 2.2, 0, 4.4, 5.3, 3.8, 6.9],
        [7.0, 3.3, 4.4, 0, 2.5, 5.1, 3.0],
        [6.4, 4.1, 5.3, 2.5, 0, 3.2, 2.7],
        [5.5, 6.0, 3.8, 5.1, 3.2, 0, 4.6],
        [8.0, 7.2, 6.9, 3.0, 2.7, 4.6, 0],
    ]

    trips = [
        [1, 2, 5],     # Trip 1 customers
        [3, 4, 6],     # Trip 2 customers
    ]

    # If your executable is not on PATH, pass lkh_path="/full/path/to/LKH"
    routes = solve_all_trips_with_lkh(
        travel_time=travel_time,
        trips=trips,
        depot=0,
        scale=1000,   # keep precision: 4.2 -> 4200
        runs=10,
        lkh_path=None,
        timeout_sec=60,
    )

    for k, r in enumerate(routes, 1):
        print(f"Trip {k} route:", r)

FileNotFoundError: Could not find LKH executable. Install LKH/LKH-3 and either:
- put it on PATH as LKH (or LKH-3), or
- set environment variable LKH_PATH=/path/to/LKH, or
- pass lkh_path=... to the solver.